In [13]:
import os
os.environ["JAVA_HOME"] = "/home/iataghav/.local/java/jdk-21.0.11+10"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ.get("PATH", "")

In [ ]:
from huggingface_hub import login

if hf_token:
    login(token=hf_token, add_to_git_credential=False)
else:
    # If no token, will prompt for manual login.
    login(add_to_git_credential=False)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.

2026-06-15 12:23:59,151 - huggingface_hub._login - WARNING - Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [15]:
import os
import sys
from pathlib import Path

def set_default_env(name, value):
    if not os.environ.get(name):
        os.environ[name] = str(value)

def find_project_root(start=Path.cwd()):
    for path in (start, *start.parents):
        if (path / 'src' / 'hyde').is_dir() and (path / 'setup.py').is_file():
            return path
    return start

ROOT_DIR = find_project_root()
os.chdir(ROOT_DIR)

src_path = str(ROOT_DIR / 'src')
set_default_env('PYTHONPATH', src_path)
if src_path not in sys.path:
    sys.path.insert(0, src_path)

set_default_env('CUDA_VISIBLE_DEVICES', '4,5,6,7')
set_default_env('GLOBAL_VISIBLE_DEVICES', os.environ['CUDA_VISIBLE_DEVICES'])
set_default_env('TOKENIZERS_PARALLELISM', 'false')

DEFAULT_NOVELHOPQA_BOOKS_ROOT = ROOT_DIR / 'novelhopqa' / 'book-corpus-root'
FALLBACK_NOVELHOPQA_BOOKS_ROOT = Path('/home/iataghav/data/passing_meta_tag/novelhopqa/book-corpus-root')
current_books_root = os.environ.get('NOVELHOPQA_BOOKS_ROOT')
if not current_books_root or not (Path(current_books_root) / 'bookmeta.json').is_file():
    if (DEFAULT_NOVELHOPQA_BOOKS_ROOT / 'bookmeta.json').is_file():
        os.environ['NOVELHOPQA_BOOKS_ROOT'] = str(DEFAULT_NOVELHOPQA_BOOKS_ROOT)
    else:
        os.environ['NOVELHOPQA_BOOKS_ROOT'] = str(FALLBACK_NOVELHOPQA_BOOKS_ROOT)

HF_CACHE_ROOT = os.environ.get('SAADI_HF_CACHE_ROOT') or '/mnt/cache/taghavi'
set_default_env('HF_HOME', HF_CACHE_ROOT)
set_default_env('HF_HUB_CACHE', Path(os.environ['HF_HOME']) / 'hub')
set_default_env('HF_DATASETS_CACHE', Path(os.environ['HF_HOME']) / 'datasets')
set_default_env('TRANSFORMERS_CACHE', Path(os.environ['HF_HOME']) / 'transformers')

print(f'ROOT_DIR={ROOT_DIR}')
print(f'CUDA_VISIBLE_DEVICES={os.environ["CUDA_VISIBLE_DEVICES"]}')
print(f'HF_HOME={os.environ["HF_HOME"]}')

from pyserini.search.faiss import FaissSearcher
from pyserini.search.lucene import LuceneSearcher
from pyserini.encode import AutoQueryEncoder
from pyserini.search import get_topics, get_qrels
from tqdm import tqdm

ROOT_DIR=/home/iataghav/SAADI_hyde
CUDA_VISIBLE_DEVICES=4,5,6,7
HF_HOME=/mnt/cache/taghavi


### Initialize Contriever Index and Query Encoder
We use [Pyserini](https://github.com/castorini/pyserini) as the search interface for the experiment. Please following the guidance in Pyserini to create Contriever index using the checkpoint from original Contriever work.

In [16]:
query_encoder = AutoQueryEncoder(encoder_dir='facebook/contriever', pooling='mean')
searcher = FaissSearcher('contriever_msmarco_index/', query_encoder)
corpus = LuceneSearcher.from_prebuilt_index('msmarco-v1-passage')

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 17420.10it/s]


### Load query and judegments for dl19-passage dataset

In [17]:
topics = get_topics('dl19-passage')
qrels = get_qrels('dl19-passage')

## Run Contriever

In [18]:
with open('dl19-contriever-top1000-trec', 'w')  as f:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            hits = searcher.search(query, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

100%|██████████| 43/43 [01:15<00:00,  1.75s/it]


In [19]:
!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage dl19-contriever-top1000-trec
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage dl19-contriever-top1000-trec
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage dl19-contriever-top1000-trec

map                   	all	0.2399
ndcg_cut_10           	all	0.4454
recall_1000           	all	0.7459


In [20]:
import os
from hyde import HuggingFaceGenerator

MODEL_NAME = 'Qwen/Qwen3-30B-A3B-Instruct-2507'
generator = HuggingFaceGenerator(MODEL_NAME, api_key=os.getenv('HF_TOKEN'), provider='auto')

def call_qwen_api(prompt: str):
    return generator.generate(prompt)

## Run HyDE

In [21]:
from time import sleep
import numpy as np
import json
from tqdm import tqdm
with open('hyde-dl19-contriever-qwen3-top1000-8rep-trec', 'w') as f, open('hyde-dl19-qwen3-gen.jsonl', 'w') as fgen:
    for qid in tqdm(topics):
        if qid in qrels:
            query = topics[qid]['title']
            print(query)
            prompt = f"""Please write a passage to answer the question
Question: {query}
Passage:"""
            for attempt in range(3):
                try:
                    contexts = [c.strip() for c in call_qwen_api(prompt)] + [query]
                    all_emb_c = []
                    for c in contexts:
                        c_emb = query_encoder.encode(c)
                        all_emb_c.append(np.array(c_emb))
                    all_emb_c = np.array(all_emb_c)
                    avg_emb_c = np.mean(all_emb_c, axis=0)
                    avg_emb_c = avg_emb_c.reshape((1, len(avg_emb_c)))
                    fgen.write(json.dumps({'query_id': qid, 'query': query, 'contexts': contexts})+'\n')
                    break
                except Exception as e:
                    if attempt == 2:
                        raise RuntimeError(f'Qwen generation failed for query id {qid}. Check HF_TOKEN and provider access.') from e
                    sleep(1)
            print(contexts)
            hits = searcher.search(avg_emb_c, k=1000)
            rank = 0
            for hit in hits:
                rank += 1
                f.write(f'{qid} Q0 {hit.docid} {rank} {hit.score} rank\n')

  0%|          | 0/43 [00:00<?, ?it/s]

how long is life cycle of flea


  0%|          | 0/43 [00:02<?, ?it/s]


RuntimeError: Qwen generation failed for query id 264014. Check HF_TOKEN and provider access.

In [ ]:
!python -m pyserini.eval.trec_eval -c -l 2 -m map dl19-passage hyde-dl19-contriever-qwen3-top1000-8rep-trec
!python -m pyserini.eval.trec_eval -c -m ndcg_cut.10 dl19-passage hyde-dl19-contriever-qwen3-top1000-8rep-trec
!python -m pyserini.eval.trec_eval -c -l 2 -m recall.1000 dl19-passage hyde-dl19-contriever-qwen3-top1000-8rep-trec


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
/home/xueguang/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar already exists!
Skipping download.
Running command: ['java', '-jar', '/home/xueguang/.cache/pyserini/eval/jtreceval-0.0.5-jar-with-dependencies.jar', '-c', '-l', '2', '-m', 'map', '/home/xueguang/.cache/pyserini/topics-and-qrels/qrels.dl19-passage.txt', 'dl19-contriever-gpt3-top1000-8rep-trec']
Results:
map                   	all	0.4039
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly se